# 간단 실습: 학생 설정 + 모델 선택 → 바로 결과

**처음에 준비 셀을 한 번 실행하세요. 이후에는 1번 설정을 바꾸고 2번 실행 셀만 실행하면 됩니다.**

- 학습: `v2_train_10800.csv` **10,800장 전체**.
- 테스트: `검증_3600장_수율숨김.csv` **3,600장 전체**.
- 채점: `검증_3600장_정답.csv`의 LOW/NORMAL 또는 수율. 정답은 학습에 넣지 않습니다.
- Colab에서는 위 CSV 세 개를 파일 패널에 업로드하세요. 로컬에서는 노트북 옆과 `해커톤_v2_데이터/data`를 찾습니다.

자동 모델 비교나 날짜별 분할 없이 **학생이 고른 모델 하나**를 학습합니다. 정답을 보고 설정을 반복 개선한 점수는 독립적인 검증 성능이 아닙니다. 상세 분석이 필요하면 [03_wafer_blind_prediction.ipynb](03_wafer_blind_prediction.ipynb)를 사용하세요.

## 준비 (처음 한 번만 실행, 코드 수정 불필요)

아래 접힌 셀은 데이터 확인·학습·채점·저장을 담당합니다. 펼치거나 수정하지 않고 실행하면 됩니다. 정답 없이 예측만 하려면 준비 코드의 `TRUTH_FILE`을 `None`으로 바꿀 수 있습니다.

In [ ]:
%pip install -q pandas numpy scikit-learn joblib

from datetime import datetime
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

TRAIN_FILE = 'v2_train_10800.csv'
TEST_FILE = '검증_3600장_수율숨김.csv'
TRUTH_FILE = '검증_3600장_정답.csv'
SENSORS = [f'S{i:02d}' for i in range(1, 9)]
CONTEXT = ['equipment_id', 'recipe_id', 'shift']
SENSOR_NAMES = ['MFC1_FLOW_SP', 'MFC1_FLOW_PV', 'CHAMBER_PRESS_PV', 'ESC_TEMP_PV', 'RF_SRC_PWR_PV', 'RF_BIAS_PWR_PV', 'CHILLER_TEMP_PV', 'APC_VALVE_POS']
MODELS = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=20, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10, class_weight='balanced_subsample', n_jobs=1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', max_iter=150, max_leaf_nodes=15, l2_regularization=1, early_stopping=False, random_state=42),
}

def find_csv(name):
    for folder in [Path.cwd(), Path('data'), Path('해커톤_v2_데이터') / 'data']:
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f'{name}을 노트북 옆이나 Colab 파일 패널에 업로드하세요.')

def validate_choices(name, values, allowed):
    if not isinstance(values, list) or not all(isinstance(value, str) for value in values):
        raise ValueError(f'{name}은 문자열 목록으로 입력하세요.')
    if set(values) - set(allowed) or len(values) != len(set(values)):
        raise ValueError(f'{name}에 잘못된 값 또는 중복이 있습니다. 선택 가능: {list(allowed)}')

def run_experiment():
    if not isinstance(MODEL, str) or MODEL not in MODELS:
        raise ValueError(f'MODEL은 {list(MODELS)} 중 하나를 선택하세요.')
    validate_choices('SENSOR_COLUMNS', SENSOR_COLUMNS, SENSORS)
    validate_choices('CONTEXT_COLUMNS', CONTEXT_COLUMNS, CONTEXT)
    if not isinstance(USE_FLOW_GAP, bool):
        raise ValueError('USE_FLOW_GAP은 True 또는 False로 입력하세요.')
    train = pd.read_csv(find_csv(TRAIN_FILE), encoding='utf-8-sig')
    blind = pd.read_csv(find_csv(TEST_FILE), encoding='utf-8-sig')
    for name, frame, count in [('학습', train, 10800), ('테스트', blind, 3600)]:
        required = ['wafer_id', 'production_date'] + SENSORS + CONTEXT
        if len(frame) != count or not set(required) <= set(frame):
            raise ValueError(f'{name} 파일의 필수 열과 {count}행을 확인하세요.')
        if frame.wafer_id.isna().any() or frame.wafer_id.duplicated().any():
            raise ValueError(f'{name}의 ID는 비어 있지 않고 서로 달라야 합니다.')
        frame['production_date'] = pd.to_datetime(frame['production_date'], errors='raise')
        frame[SENSORS] = frame[SENSORS].apply(pd.to_numeric, errors='raise')
        if frame.production_date.isna().any() or frame[CONTEXT].isna().any().any():
            raise ValueError(f'{name} 날짜 또는 생산 맥락에 빈 값이 있습니다.')
        if np.isinf(frame[SENSORS].to_numpy()).any() or frame[SENSORS].isna().all().any():
            raise ValueError(f'{name} 센서에 무한대 또는 전부 빈 열이 있습니다.')
    if 'yield_rate' not in train:
        raise ValueError('학습 파일에는 yield_rate가 필요합니다.')
    train['yield_rate'] = pd.to_numeric(train['yield_rate'], errors='raise')
    if not train.yield_rate.between(0, 100).all():
        raise ValueError('학습 수율은 빈 값 없이 0~100 사이여야 합니다.')
    if {'yield_rate', 'yield_status', 'cause'} & set(blind):
        raise ValueError('테스트에는 정답 열이 없는 수율숨김 파일을 사용하세요.')
    if set(train.wafer_id) & set(blind.wafer_id) or train.production_date.max() >= blind.production_date.min():
        raise ValueError('학습·테스트 ID는 겹치지 않고 테스트 날짜는 학습보다 뒤여야 합니다.')
    conditions = {'equipment_id': FOCUS_EQUIPMENT, 'recipe_id': FOCUS_RECIPE, 'shift': FOCUS_SHIFT}
    for column, values in conditions.items():
        validate_choices(f'{column} 조건', values, sorted(train[column].unique()))
    numeric = SENSOR_COLUMNS + (['flow_gap'] if USE_FLOW_GAP else []) + [f'is_{column}' for column, values in conditions.items() if values]
    features = numeric + CONTEXT_COLUMNS
    if not features:
        raise ValueError('센서, 유량 차이, 생산 맥락 또는 조건 중 입력을 최소 하나 선택하세요.')

    def inputs(frame):
        result = frame[SENSOR_COLUMNS + CONTEXT_COLUMNS].copy()
        if USE_FLOW_GAP:
            result['flow_gap'] = frame['S02'] - frame['S01']
        for column, values in conditions.items():
            if values:
                result[f'is_{column}'] = frame[column].isin(values).astype(int)
        return result[features]

    X_train, X_test = inputs(train), inputs(blind)
    y_train = train.yield_rate.lt(92).astype(int)
    if y_train.nunique() != 2 or X_train[numeric].isna().all().any():
        raise ValueError('학습에는 정상·저수율이 모두 필요하며 전부 빈 입력은 사용할 수 없습니다.')
    preprocess = ColumnTransformer([
        ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
        ('context', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CONTEXT_COLUMNS),
    ])
    model = make_pipeline(preprocess, clone(MODELS[MODEL])).fit(X_train, y_train)
    predictions = blind[['wafer_id']].copy()
    predictions['risk_score'] = model.predict_proba(X_test)[:, 1]
    predictions['predicted_low'] = predictions.risk_score.ge(0.5).astype(int)
    predictions = predictions.sort_values(['risk_score', 'wafer_id'], ascending=[False, True]).reset_index(drop=True)
    predictions.insert(1, 'inspection_rank', np.arange(1, 3601))
    submission = predictions.head(100)[['wafer_id']].copy()
    report = {'model': MODEL, 'training_rows': len(train), 'test_rows': len(blind), 'inspection_count': len(submission)}
    evaluation = None
    if TRUTH_FILE is not None:
        truth = pd.read_csv(find_csv(TRUTH_FILE), encoding='utf-8-sig').rename(columns=dict(zip(SENSOR_NAMES, SENSORS)))
        if 'wafer_id' not in truth or not {'yield_status', 'yield_rate'} & set(truth):
            raise ValueError('정답에는 wafer_id와 yield_status 또는 yield_rate가 필요합니다.')
        if len(truth) != 3600 or truth.wafer_id.nunique() != 3600 or set(truth.wafer_id) != set(blind.wafer_id):
            raise ValueError('정답 ID 3,600개가 테스트 ID와 정확히 일치해야 합니다.')
        aligned = truth.set_index('wafer_id').loc[blind.wafer_id]
        for sensor in SENSORS:
            if sensor in truth and not np.allclose(pd.to_numeric(aligned[sensor], errors='raise'), blind[sensor], rtol=0, atol=1e-8, equal_nan=True):
                raise ValueError(f'정답의 {sensor} 값이 테스트와 다릅니다. 같은 버전의 정답을 사용하세요.')
        for column in CONTEXT:
            if column in truth and aligned[column].tolist() != blind[column].tolist():
                raise ValueError(f'정답의 {column} 값이 테스트와 다릅니다.')
        if 'production_date' in truth and pd.to_datetime(aligned.production_date, errors='raise').tolist() != blind.production_date.tolist():
            raise ValueError('정답의 생산 날짜가 테스트와 다릅니다.')
        actual = None
        if 'yield_rate' in truth:
            truth['yield_rate'] = pd.to_numeric(truth.yield_rate, errors='raise')
            if not truth.yield_rate.between(0, 100).all():
                raise ValueError('정답 수율은 빈 값 없이 0~100 사이여야 합니다.')
            actual = truth.yield_rate.lt(92).astype(int)
        if 'yield_status' in truth:
            if not truth.yield_status.isin(['LOW', 'NORMAL']).all():
                raise ValueError('정답 yield_status는 LOW 또는 NORMAL이어야 합니다.')
            status = truth.yield_status.eq('LOW').astype(int)
            if actual is not None and not actual.equals(status):
                raise ValueError('정답 수율과 yield_status가 일치하지 않습니다.')
            actual = status
        truth['actual_low'] = actual
        columns = ['wafer_id', 'actual_low'] + [column for column in ['yield_status', 'yield_rate'] if column in truth]
        evaluation = predictions.merge(truth[columns], on='wafer_id', validate='one_to_one')
        hits = int(evaluation.head(100).actual_low.sum())
        total_low = int(evaluation.actual_low.sum())
        report.update({
            'hits_at_100': hits, 'precision_at_100': hits / 100,
            'normal_selected': 100 - hits, 'missed_low': total_low - hits,
            'overall_accuracy': float(evaluation.predicted_low.eq(evaluation.actual_low).mean()),
            'all_normal_baseline_accuracy': 1 - total_low / 3600,
        })
        if total_low:
            report['recall_at_100'] = hits / total_low
        else:
            print('정답의 저수율이 0장이므로 재현율은 정의되지 않습니다.')
    output = Path('outputs') / datetime.now().strftime('simple_%Y%m%d_%H%M%S_%f')
    output.mkdir(parents=True, exist_ok=False)
    submission.to_csv(output / '검증_우선검사목록.csv', index=False, encoding='utf-8-sig')
    predictions.to_csv(output / '검증_전체예측.csv', index=False, encoding='utf-8-sig')
    if evaluation is not None:
        evaluation.to_csv(output / '검증_전체정답비교.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame([report]).to_csv(output / '실험_결과.csv', index=False, encoding='utf-8-sig')
    joblib.dump(model, output / '저수율_예측모델.joblib')
    criteria = {'model': MODEL, 'seed': 42, 'features': features, 'sensor_columns': SENSOR_COLUMNS, 'context_columns': CONTEXT_COLUMNS, 'use_flow_gap': USE_FLOW_GAP, 'conditions': conditions, 'yield_cutoff': 92, 'reference_score_cutoff': 0.5, 'inspection_count': 100, 'ranking': 'risk_score descending, wafer_id ascending'}
    (output / '판별기준.json').write_text(json.dumps(criteria, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'모델: {MODEL} | 학습: 10,800장 | 테스트: 3,600장 | 검사 대상: 100장')
    if evaluation is not None:
        print(f'100장 적중률: {hits}% ({hits}/100장) | 전체 정확도: {report["overall_accuracy"]:.2%}')
        print(f'함께 고른 정상: {100 - hits}장 | 놓친 저수율: {total_low - hits}장')
        print(f'모두 정상으로 예측한 정확도: {report["all_normal_baseline_accuracy"]:.2%} → 과제에서는 100장 적중률을 우선하세요.')
    else:
        print('정답 채점을 건너뛰었습니다. 실제 적중률은 아직 계산하지 않았습니다.')
    display((evaluation if evaluation is not None else predictions).head(10).round(4))
    print(f'저장 폴더: {output.resolve()}')
    print('제출 파일: 검증_우선검사목록.csv')
    return {'model': model, 'features': features, 'predictions': predictions, 'submission': submission, 'report': report, 'output_dir': output}


## 1. 학생 설정과 모델 선택

**아래 값만 바꾸세요.** MODEL은 `LogisticRegression`, `DecisionTree`, `RandomForest`, `HistGradientBoosting` 중 하나입니다.

- 센서 예: `['S01', 'S02', 'S08']`. 생산 맥락 예: `['equipment_id', 'recipe_id']`.
- `FOCUS_EQUIPMENT = ['ETCH-C', 'ETCH-D']`처럼 넣으면 해당 장비인지 0/1 입력으로 추가합니다. 레시피는 R1/R2/R3, 근무조는 DAY/NIGHT입니다.
- `[]`는 사용하지 않음, `USE_FLOW_GAP`은 유량 차이(S02−S01) 사용 여부입니다. 유량 차이는 S01/S02를 센서 목록에 넣지 않아도 계산됩니다.
- 장비·레시피 조건은 **행을 제외하지 않습니다.** 조건을 넣었다고 그 조건이 반드시 불량 원인이라는 뜻도 아닙니다.

In [ ]:
MODEL = 'HistGradientBoosting'

SENSOR_COLUMNS = SENSORS.copy()
CONTEXT_COLUMNS = CONTEXT.copy()
USE_FLOW_GAP = True

FOCUS_EQUIPMENT = []
FOCUS_RECIPE = []
FOCUS_SHIFT = []

## 2. 실행 → 결과 확인

설정 셀을 실행한 뒤 아래 셀을 실행하세요. 한 번에 **학습 → 3,600장 예측 → 정답 채점 → 검사 100장 저장**까지 진행합니다. 다시 실험하려면 설정 셀과 이 셀만 다시 실행하면 됩니다. 매번 새 결과 폴더를 만들어 이전 목록은 덮어쓰지 않습니다.

**제출 파일은 `검증_우선검사목록.csv`입니다.** Colab에서는 출력된 폴더를 왼쪽 파일 패널에서 열어 파일을 다운로드하세요. 모델 점수는 보정된 실제 불량 확률이 아니라 검사 순서를 정할 위험 점수입니다.

In [ ]:
result = run_experiment()